# Debugging a failing run with telemetry

This notebook **fails on purpose**. It runs a two-member fusion where one member is a model the
gateway cannot serve, then walks from the symptom to the root cause:

1. run the fusion;
2. read the symptom in the report;
3. find the run's trace id;
4. read the SDK telemetry: failing spans, ERROR logs, usage;
5. follow the trace id into the gateway logs;
6. fix the recipe.

It runs on `replay` only: the healthy member is served from the recorded ifeval cache, and the
unservable member is a route the tape never recorded, so it fails the same way every time and
never reaches a provider. The workbench passes it only when it fails exactly as its metadata
declares (`workbench.expect`) and the evidence is complete: the trace chain, and the error in
the telemetry.

In [ ]:
ENGINE_URL = "http://127.0.0.1:9108"
SCOREBOARD_URL = "http://127.0.0.1:9106"  # always explicit; replay has none and never calls it
BENCHMARK = "ifeval"
LIMIT = 1
UNSERVABLE_MODEL = "openrouter/openai/gpt-5.5"  # a declared route the ifeval tape never recorded
EXPECTED_CODE = "upstream_error"  # discovered on replay, never guessed
EXPECTED_STAGE = "candidate"
EVENTS_PATH = ""  # set by the workbench: SDK events (and the report files beside them)
GATEWAY_LOG = ""  # optional: the stack's aigateway.log, to filter for this run's lines

In [ ]:
import dataclasses
import json
import re
from collections import Counter
from collections.abc import Mapping
from pathlib import Path

import screamingface as sf

if not SCOREBOARD_URL:  # spec §3.1: never fall back to the hosted leaderboard
    raise ValueError("SCOREBOARD_URL must be explicit")
client = sf.Client(engine_url=ENGINE_URL, scoreboard_url=SCOREBOARD_URL)

## 1. Run the fusion

Two members answer, one synthesizer combines. The first member is exactly the request the
ifeval tape recorded, so the replay gateway serves it from cache. The second member is
`UNSERVABLE_MODEL`: the Engine knows the route, but the replay gateway holds no answer and no
key for it, so it refuses the call (`404 profile_not_found`) without any live request.

In [ ]:
HEALTHY_MODEL = "openrouter/mistralai/ministral-3b-2512"  # the golden's first member: a cache hit
candidate = sf.Fusion(
    [
        sf.Model(
            "openrouter/mistralai/ministral-3b-2512",
            prompt=(
                "Answer the request accurately and completely. Follow every "
                "instruction and formatting constraint in the request."
            ),
            params={"max_tokens": 8192, "temperature": 0.0},
        ),
        sf.Model(
            UNSERVABLE_MODEL,
            prompt=(
                "Answer the request accurately and completely. Follow every "
                "instruction and formatting constraint in the request."
            ),
        ),
    ],
    name="debug_unservable",
    synthesizer=sf.Model(
        "openrouter/deepseek/deepseek-v4-flash",
        prompt=(
            "Produce one final answer to the original request from the panel "
            "drafts. Preserve every instruction and formatting constraint."
        ),
        params={"max_tokens": 8192, "temperature": 0.0},
    ),
)
candidate

In [ ]:
recorder = None
if EVENTS_PATH:  # the workbench records every SDK event, redacted, into the run's bundle
    from workbench.telemetry_recorder import EventRecorder

    recorder = EventRecorder(EVENTS_PATH)
seen = []  # outside the workbench, the same events in memory


def on_event(event):
    seen.append(event)
    if recorder is not None:
        recorder(event)


report = None
evaluate_error = None
try:
    report = client.evaluate(
        candidate, benchmark=BENCHMARK, limit=LIMIT, progress=False, on_event=on_event
    )
except sf.ScreamingFaceError as exc:
    evaluate_error = exc
    report = getattr(exc, "partial_report", None)  # ExecutionError: the Candidates that ran

### Record the evidence first

Before any check, the next cell reports the run's trace ids and failure codes to the workbench
(the single `WORKBENCH_RESULT` line). If a later check fails, the evidence of the run that
misbehaved is already in the bundle.

In [ ]:
ran = list(report.candidates) if report is not None else []
error_trace_id = evaluate_error.trace_id if evaluate_error is not None else None
trace_ids = [c.trace_id for c in ran if c.trace_id] + [error_trace_id]
failure_codes = sorted(
    {f.code for c in ran for f in c.failures}
    | {f.code for c in ran for case in c.cases for f in case.failures}
)
print(
    "WORKBENCH_RESULT="
    + json.dumps(
        {
            "trace_ids": [t for t in dict.fromkeys(trace_ids) if t],
            "run_ids": [str(c.run_id) for c in ran],
            "failure_codes": failure_codes,
            "evaluate_error": None
            if evaluate_error is None
            else {"type": type(evaluate_error).__name__, "code": evaluate_error.code},
        }
    )
)
if recorder is not None:  # report.json + accounting.json beside the events; never raises
    recorder.finish(report)

## 2. Symptom: the report

`client.evaluate()` did **not** raise, and the run terminated `succeeded`: a member failing is a
case failure, not a run failure. The report says which case failed, at which stage and with which
code. The candidate has no score, because its only case has no output: with the default
`quorum="all"` one failed member fails the whole case before the synthesizer runs. So the
healthy member's answer is not in the report; its cache hit is, and section 4 shows its usage.

Only the code, stage and source code are printed. A failure's `message` is provider text, which on
a live target can quote request details, so it is never printed here.

In [ ]:
if evaluate_error is not None:
    raise evaluate_error  # this recipe fails inside the run; evaluate() itself must not raise
(result,) = list(report.candidates)
failed = [case for case in result.cases if case.failures]
for case in failed:
    for failure in case.failures:
        source = dict(failure.metadata.get("source_error") or {})
        print(
            f"case {case.case_id}: status={case.status} stage={failure.stage} "
            f"code={failure.code} retryable={failure.retryable} "
            f"source_code={source.get('source_code')}"
        )
print(f"score={result.score} cache_hits={result.cache_hits}")
observed = sorted({(f.code, f.stage) for case in failed for f in case.failures})
assert observed, "unexpected pass: no case failed"
assert observed == [(EXPECTED_CODE, EXPECTED_STAGE)], observed
assert result.cache_hits >= 1, "the healthy member was not served from the cache"

## 3. The trace id

Every run has one trace id. It is on each candidate of the report (`result.trace_id`). When
`client.evaluate()` raises after the run started, the SDK error carries it instead
(`exc.trace_id`). The engine, the gateway and the recorded spans all carry the same id, so it is
the key to everything that follows.

In [ ]:
trace_id = result.trace_id
assert trace_id and re.fullmatch(r"[0-9a-f]{32}", trace_id), trace_id
print("trace id:", trace_id)

## 4. Telemetry: what the SDK saw

The SDK streams events while the run executes: `Span` (url4 nodes), `Log`, `Usage`. The
workbench records them, redacted, into the run's bundle (`events.jsonl`); the cell below loads
that copy. Two filters find the error:

- **failing spans**: `status == "error"` (or a refusal);
- **ERROR logs**: `severity_number >= 17`.

Read them together. The failing spans are the url4 nodes the failure propagated through, and the
model-bearing one names the **healthy** member (its call succeeded, from cache). Only the ERROR
`Log` "Model call failed" names the failing model. Its `sf.activity.failure_code` is the
activity's own code, not the report's: trust the report for the code, and the log for the model.

In [ ]:
def as_row(value):
    """An in-memory SDK event as the same plain tree the recorder writes."""
    if dataclasses.is_dataclass(value) and not isinstance(value, type):
        row = {f.name.lstrip("_"): getattr(value, f.name) for f in dataclasses.fields(value)}
        value = {"kind": value.kind, **row} if hasattr(value, "kind") else row
    if isinstance(value, Mapping):
        return {str(k): as_row(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [as_row(v) for v in value]
    return value


if EVENTS_PATH:  # the workbench's recorded copy, already redacted
    lines = Path(EVENTS_PATH).read_text(encoding="utf-8").splitlines()
    events = [json.loads(line) for line in lines if line.strip()]
else:
    events = [as_row(event) for event in seen]
print(f"{len(events)} events:", dict(Counter(e["kind"] for e in events)))

failing_spans = [
    e for e in events if e["kind"] == "span" and (e.get("status") == "error" or e.get("refusal"))
]
error_logs = [e for e in events if e["kind"] == "log" and e.get("severity_number", 0) >= 17]
for span in failing_spans:
    name = str(span.get("name"))
    name = name if len(name) <= 48 else name[:45] + "..."  # url4 map nodes name the expression
    print(
        f"span {name} [{span.get('operation')}]: status={span.get('status')} "
        f"provider={span.get('provider')} model={span.get('request_model')} "
        f"finish={list(span.get('finish_reasons') or [])} cache={span.get('cache_status')}"
    )
for log in error_logs:
    attrs = log.get("attributes") or {}
    print(
        f"{log.get('severity_text')} {log.get('body')!r}: "
        f"model={attrs.get('sf.activity.model_id')} kind={attrs.get('sf.activity.kind')} "
        f"failure_code={attrs.get('sf.activity.failure_code')}"
    )
assert error_logs, "no ERROR log: the telemetry missed the failure"
failing_models = {(log.get("attributes") or {}).get("sf.activity.model_id") for log in error_logs}
assert UNSERVABLE_MODEL in failing_models, failing_models

`Usage` events are the accounting per model. A `Usage` row for the healthy member proves it ran
even though its answer never reached the report. On replay its counts and cost are zero: the
answer came from the cache.

In [ ]:
usage = [e for e in events if e["kind"] == "usage"]
for row in usage:
    amounts = row.get("usage") or {}
    print(
        f"usage scope={row.get('scope')} provider={row.get('provider')} "
        f"model={row.get('model')} input_tokens={amounts.get('input_tokens')} "
        f"output_tokens={amounts.get('output_tokens')} cost_usd={amounts.get('cost_usd')}"
    )
assert any(row.get("model") == HEALTHY_MODEL for row in usage), "no Usage for the healthy member"

## 5. Follow the trace into the logs

The engine and the gateway write `key=value` log lines. Each gateway line of this run carries
`trace_id=<id>` and a `gateway_call_id=call_<id>`, one call id per member call. The workbench's
bundle page for this run (`trace.json`, Telemetry) lists those lines and links each failing span
and ERROR log to them.

To filter a log yourself, match the **exact token** `trace_id=<32 hex>`, bounded by whitespace,
the same rule the workbench verdict uses. A substring match is wrong: it also picks up
`parent_trace_id=<id>`, or a longer id that merely starts with this one, and those lines belong to
another run.

On this failure the gateway logs a "usage accounting started" line for **both** member calls,
and a "global cache hit" line only for the healthy one. The call id with no cache-hit line is the
refused member: the 404 itself logs nothing at the gateway. The excerpt below has that shape,
plus decoys a substring match would wrongly keep. Set `GATEWAY_LOG` to the stack's
`aigateway.log` to filter the real one.

In [ ]:
TRACE_ID_TOKEN = re.compile(r"(?<!\S)trace_id=([0-9a-f]{32})(?!\S)")  # the verdict's rule


def lines_of(trace_id, text):
    """The lines carrying an exact `trace_id=<id>` token for this id, never a substring."""
    return [line for line in text.splitlines() if trace_id in TRACE_ID_TOKEN.findall(line)]


other = "0" * 32
excerpt = "\n".join(
    [
        f"INFO session gateway_call_id=call_a trace_id={trace_id} usage accounting started",
        f"INFO session gateway_call_id=call_b trace_id={trace_id} usage accounting started",
        f"INFO chat_cache_stage gateway_call_id=call_a trace_id={trace_id} global cache hit "
        f"model='{HEALTHY_MODEL}'",
        f"INFO session gateway_call_id=call_c parent_trace_id={trace_id} another run",
        f"INFO session gateway_call_id=call_d trace_id={trace_id}ff another run",
        f"INFO session gateway_call_id=call_e trace_id={other} another run",
    ]
)
mine = lines_of(trace_id, excerpt)
substring = [line for line in excerpt.splitlines() if trace_id in line]
# WHY this wording: "token: <n>" reads like a credential to the redactor and gets masked.
print(f"lines kept by the exact rule: {len(mine)}; by a substring match: {len(substring)}")
calls = {}
for line in mine:
    call = re.search(r"(?<!\S)gateway_call_id=(\S+)", line).group(1)
    calls.setdefault(call, []).append("cache hit" if "cache hit" in line else "started")
for call, seen_as in calls.items():
    verdict = "served" if "cache hit" in seen_as else "refused: no cache-hit line"
    print(f"{call}: {', '.join(seen_as)} -> {verdict}")
if GATEWAY_LOG and Path(GATEWAY_LOG).is_file():
    print(f"--- {GATEWAY_LOG}, this run only")
    for line in lines_of(trace_id, Path(GATEWAY_LOG).read_text(encoding="utf-8")):
        print(line)

In [ ]:
assert len(mine) == 3 and len(substring) == 5, (mine, substring)
assert sorted(calls) == ["call_a", "call_b"], calls

## 6. Root cause and the fixed recipe

**Root cause:** `UNSERVABLE_MODEL` is a route the gateway cannot serve here (on replay: no
recorded answer and no key, so a `404 profile_not_found`). The engine reports it as
`upstream_error` at stage `candidate`, and with `quorum="all"` it takes the whole case down.

**Fix:** use members the gateway can serve. Here the unservable member is replaced by the other
member the ifeval golden recorded. It is left as text, not run, so this notebook stays failing by
design.

```python
candidate = sf.Fusion(
    [
        sf.Model(
            "openrouter/mistralai/ministral-3b-2512",
            prompt=(
                "Answer the request accurately and completely. Follow every "
                "instruction and formatting constraint in the request."
            ),
            params={"max_tokens": 8192, "temperature": 0.0},
        ),
        sf.Model(
            "openrouter/microsoft/phi-4",
            prompt=(
                "Answer the request accurately and completely. Follow every "
                "instruction and formatting constraint in the request."
            ),
            params={"max_tokens": 8192, "temperature": 0.0},
        ),
    ],
    name="debug_fixed",
    synthesizer=sf.Model(
        "openrouter/deepseek/deepseek-v4-flash",
        prompt=(
            "Produce one final answer to the original request from the panel "
            "drafts. Preserve every instruction and formatting constraint."
        ),
        params={"max_tokens": 8192, "temperature": 0.0},
    ),
)
```

## 7. The result

The `WORKBENCH_RESULT` line printed after section 1 carries the trace id and the failure codes.
The workbench checks them against this notebook's declaration (`workbench.expect`): the declared
code at the declared stage, the trace in the engine and gateway logs with call ids, and an error
in the telemetry. Anything else, including an unexpected pass, is red.

In [ ]:
client.close()